In [ ]:
# ============================================================
# COLAB TRANSFER - 02: FILA AIR/URL CIVITAI -> STAGING
# ============================================================
# A entrada preferida é AIR; URLs Civitai continuam aceitas.
# Todos os downloads são sequenciais e ficam no staging até o upload.

import os
import shutil
import subprocess
import sys
from pathlib import Path

WORKDIR = Path('/content/colab-pipeline')
SCRIPTS_DIR = WORKDIR / 'scripts'
REPO_URL = 'https://github.com/automadevs/colab-pipeline.git'
if WORKDIR.exists():
    subprocess.run(['git', '-C', str(WORKDIR), 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORKDIR)], check=True)
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

MANAGER = SCRIPTS_DIR / 'kaggle_dataset_manager.py'
if not MANAGER.exists():
    raise FileNotFoundError(f'Módulo não encontrado: {MANAGER}')

from kaggle_dataset_manager import get_secret, download_input_queue, write_manifest

TOKEN = get_secret('CIVITAI_TOKEN') or get_secret('CIVITAI_API_KEY')
if not TOKEN:
    raise RuntimeError('CIVITAI_TOKEN não encontrado nos Secrets/env do Colab')

DATASET = 'automamermaid/comfydocs'
STAGING_DIR = Path('/content/kaggle_staging')
STAGING_DIR.mkdir(parents=True, exist_ok=True)

items = download_input_queue(STAGING_DIR, TOKEN)
if not items:
    raise ValueError('Nenhum arquivo foi adicionado ao staging.')

write_manifest(STAGING_DIR / 'dataset-manifest.json', DATASET, {item.path: item for item in items})
print(f'\n[FILA CONCLUÍDA] {len(items)} arquivo(s) no staging: {STAGING_DIR}')
print('Próximo passo: execute 03_upload_kaggle.ipynb para revisar e publicar.')